# Сервис интеллектуального анализа эффективности физических упражнений

Ноутбук сохраняет исходную логику подготовки временных окон и дополняет её тремя моделями регрессии:

- линейная регрессия как базовая точка сравнения;
- MLP для работы с длинными векторами признаков;
- LSTM для работы с последовательностями через тензоры.

В этом варианте используется сценарий из диплома: по **14 последним дням** наблюдений прогнозируется изменение `fitness_level` на **следующие 14 дней**.

**1) Импорт библиотек и объявление основных переменных:**

In [ ]:
import os
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")

# Пути к данным
CSV_PATH = "./health_fitness_dataset.csv"

# Основные колонки датасета
USER_COL = "participant_id"
DATE_COL = "date"
TARGET_COL = "fitness_level"

# Сценарий из работы:
# по 14 последним дням прогнозируем изменение fitness_level на следующие 14 дней
HORIZON_DAYS = 14
L = 14

# Кандидаты на one-hot кодирование
CATEGORICAL_CANDIDATES = [
    "gender",
    "activity_type",
    "intensity",
    "smoking_status",
    "health_condition",
]

# Настройки обучения
RANDOM_STATE = 42
BATCH_SIZE = 64
LSTM_EPOCHS = 30
LINEAR_EPOCHS = 60
MLP_EPOCHS = 60
LEARNING_RATE = 1e-3

# Аппаратная конфигурация
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

def seed_everything(seed: int = RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

**2) Чтение датасета и первый этап обработки:**

In [ ]:
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"Файл датасета не найден по пути: {CSV_PATH}\n"
        "Положите CSV рядом с ноутбуком или поменяйте переменную CSV_PATH."
    )

df = pd.read_csv(CSV_PATH)

# преобразование дат
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df = df.dropna(subset=[USER_COL, DATE_COL, TARGET_COL])

# сортировка: необходима для merge_asof и окон
df = df.sort_values([USER_COL, DATE_COL]).reset_index(drop=True)

print("Размер исходного датасета:", df.shape)
display(df.head(100))

**3) Подготовка целевой переменной:**

In [ ]:
# создание колонки с датой через HORIZON_DAYS относительно текущей
df["future_date"] = df[DATE_COL] + pd.to_timedelta(HORIZON_DAYS, unit="D")

# создание таблицы с будущими измерениями
future_tbl = df[[USER_COL, DATE_COL, TARGET_COL]].rename(
    columns={DATE_COL: "future_meas_date", TARGET_COL: "fitness_level_future"}
).sort_values([USER_COL, "future_meas_date"])

# добавление к каждой текущей записи ближайшего будущего измерения того же человека
df = pd.merge_asof(
    left=df.sort_values([USER_COL, "future_date"]),
    right=future_tbl,
    left_on="future_date",
    right_on="future_meas_date",
    by=USER_COL,
    direction="forward",
    allow_exact_matches=True,
)

# удаление строк, для которых не нашлось будущего измерения
df = df.dropna(subset=["fitness_level_future"]).reset_index(drop=True)

# target: абсолютное изменение fitness_level за горизонт HORIZON_DAYS
df["y_delta"] = df["fitness_level_future"] - df[TARGET_COL]

# дополнительная бинарная метка роста
df["y_grow"] = (df["y_delta"] > 0).astype("int64")

print("Размер после формирования target:", df.shape)
display(df[[USER_COL, DATE_COL, TARGET_COL, "fitness_level_future", "y_delta", "y_grow"]].head())

**4) Преобразование всех данных в числа и one-hot:**

In [ ]:
# удаление target-колонок, чтобы они не попали в признаки
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
drop_num = {TARGET_COL, "fitness_level_future", "y_delta", "y_grow"}
num_cols = [c for c in num_cols if c not in drop_num]

# сопоставление ожидаемых категориальных колонок и реально существующих
cat_cols = [c for c in CATEGORICAL_CANDIDATES if c in df.columns]

# датафрейм с id, датой и выбранными признаками
feat_df = df[[USER_COL, DATE_COL] + num_cols + cat_cols].copy()

# one-hot кодирование категорий
if cat_cols:
    feat_df = pd.get_dummies(feat_df, columns=cat_cols, dummy_na=True)

# список финальных признаковых колонок
row_feature_cols = [c for c in feat_df.columns if c not in [USER_COL, DATE_COL]]
N_FEATURES_PER_STEP = len(row_feature_cols)

print("Количество признаков на один день:", N_FEATURES_PER_STEP)
print("Первые признаки:", row_feature_cols[:20])

**5) Оконное представление данных (временной отрезок в один длинный вектор):**

In [ ]:
def build_windows_for_user(user_block: pd.DataFrame):
    # внутри пользователя обязательно держим строгий хронологический порядок
    user_block = user_block.sort_values(DATE_COL)

    # матрица признаков по дням
    ub_feat = feat_df.loc[user_block.index, row_feature_cols].to_numpy(dtype=np.float32)

    # targets и даты
    y_delta = user_block["y_delta"].to_numpy(dtype=np.float32)
    y_grow = user_block["y_grow"].to_numpy(dtype=np.int64)
    current_fitness = user_block[TARGET_COL].to_numpy(dtype=np.float32)
    future_fitness = user_block["fitness_level_future"].to_numpy(dtype=np.float32)
    dates = user_block[DATE_COL].to_numpy()

    # если данных о пользователе меньше длины окна, пропускаем
    if len(user_block) < L:
        return None

    X_list, yd_list, yg_list, cf_list, ff_list, d_list = [], [], [], [], [], []

    # перебор всех конечных позиций окна
    for i in range(L - 1, len(user_block)):
        window = ub_feat[i - L + 1 : i + 1]

        # Для линейной модели и MLP сохраняем длинный вектор
        X_list.append(window.reshape(-1))

        # target относится к последнему дню окна
        yd_list.append(y_delta[i])
        yg_list.append(y_grow[i])
        cf_list.append(current_fitness[i])
        ff_list.append(future_fitness[i])
        d_list.append(dates[i])

    return (
        np.stack(X_list),
        np.array(yd_list, dtype=np.float32),
        np.array(yg_list, dtype=np.int64),
        np.array(cf_list, dtype=np.float32),
        np.array(ff_list, dtype=np.float32),
        np.array(d_list),
    )

Xs, yds, ygs, current_fs, future_fs, gids, end_dates = [], [], [], [], [], [], []

# собираем окна отдельно по каждому пользователю, чтобы не смешивать разные временные ряды
for pid, g in df.groupby(USER_COL, sort=False):
    out = build_windows_for_user(g)
    if out is None:
        continue

    Xg, ydg, ygg, cfg, ffg, dg = out
    Xs.append(Xg)
    yds.append(ydg)
    ygs.append(ygg)
    current_fs.append(cfg)
    future_fs.append(ffg)
    gids.append(np.full(len(ydg), pid, dtype=object))
    end_dates.append(dg)

# если после фильтрации не удалось собрать ни одного окна, дальше обучать модели нельзя
if not Xs:
    raise ValueError(
        "Не удалось сформировать ни одного временного окна. "
        "Проверьте, что после подготовки target у пользователей осталось минимум L наблюдений."
    )

# объединяем окна всех пользователей в единые массивы
X = np.concatenate(Xs, axis=0)
y_delta = np.concatenate(yds, axis=0)
y_grow = np.concatenate(ygs, axis=0)
current_fitness_arr = np.concatenate(current_fs, axis=0)
future_fitness_arr = np.concatenate(future_fs, axis=0)
groups = np.concatenate(gids, axis=0)
end_dates_arr = np.concatenate(end_dates, axis=0)

print("Prepared windowed data:")
print("X:", X.shape, "(n_samples, L * F)")
print("y_delta:", y_delta.shape)
print("groups:", groups.shape)

**6) Стандартизация и split на train/test по людям:**

In [ ]:
# split по группам корректен только если осталось хотя бы два разных пользователя
unique_groups = np.unique(groups)
if len(unique_groups) < 2:
    raise ValueError("Для train/test split нужно минимум 2 разных пользователя после построения окон.")

# делим выборку по людям, чтобы окна одного пользователя не попадали одновременно в train и test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y_delta, groups=groups))

X_train_raw, X_test_raw = X[train_idx], X[test_idx]
y_delta_train, y_delta_test = y_delta[train_idx], y_delta[test_idx]
y_grow_train, y_grow_test = y_grow[train_idx], y_grow[test_idx]
current_fit_train, current_fit_test = current_fitness_arr[train_idx], current_fitness_arr[test_idx]
future_fit_train, future_fit_test = future_fitness_arr[train_idx], future_fitness_arr[test_idx]

# стандартизация для моделей, работающих с плоскими признаками
flat_scaler = StandardScaler()
X_train = flat_scaler.fit_transform(X_train_raw).astype(np.float32)
X_test = flat_scaler.transform(X_test_raw).astype(np.float32)

print("Train:", X_train.shape, y_delta_train.shape)
print("Test :", X_test.shape, y_delta_test.shape)

**7) Подготовка последовательностей и тензоров для LSTM:**

In [ ]:
# Из длинного вектора обратно в последовательность формы (samples, L, F)
X_train_seq = X_train.reshape(-1, L, N_FEATURES_PER_STEP).astype(np.float32)
X_test_seq = X_test.reshape(-1, L, N_FEATURES_PER_STEP).astype(np.float32)

# Тензоры нужны именно для LSTM
X_train_tensor = torch.tensor(X_train_seq, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_seq, dtype=torch.float32)
y_train_tensor = torch.tensor(y_delta_train.reshape(-1, 1), dtype=torch.float32)
y_test_tensor = torch.tensor(y_delta_test.reshape(-1, 1), dtype=torch.float32)

# Оборачиваем тензоры в Dataset/DataLoader для батчевого обучения
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print("LSTM train tensor:", tuple(X_train_tensor.shape))
print("LSTM test tensor :", tuple(X_test_tensor.shape))

**8) Вспомогательные функции оценки качества:**

In [ ]:
# считаем базовые регрессионные метрики в одном месте, чтобы одинаково сравнивать модели
def regression_metrics(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": rmse,
        "R2": r2_score(y_true, y_pred),
    }

# компактный вывод словаря метрик в удобном виде
def print_metrics(title, metrics_dict):
    print(title)
    for k, v in metrics_dict.items():
        print(f"  {k}: {v:.6f}")

**9) Линейная регрессия на готовых данных:**

In [ ]:
# Для визуализации loss используем линейную регрессию, обучаемую SGD
linear_model = SGDRegressor(
    loss="squared_error",
    penalty=None,
    learning_rate="invscaling",
    eta0=0.01,
    power_t=0.25,
    max_iter=1,
    tol=None,
    random_state=RANDOM_STATE,
    warm_start=True,
)

# будем сохранять MSE на train/test после каждой эпохи
linear_train_loss = []
linear_test_loss = []

for epoch in range(LINEAR_EPOCHS):
    # max_iter=1 + warm_start=True позволяют вручную пройтись по эпохам
    linear_model.fit(X_train, y_delta_train)

    pred_train = linear_model.predict(X_train)
    pred_test = linear_model.predict(X_test)

    linear_train_loss.append(mean_squared_error(y_delta_train, pred_train))
    linear_test_loss.append(mean_squared_error(y_delta_test, pred_test))

# финальные предсказания после завершения обучения
linear_pred_train = linear_model.predict(X_train)
linear_pred_test = linear_model.predict(X_test)

linear_train_metrics = regression_metrics(y_delta_train, linear_pred_train)
linear_test_metrics = regression_metrics(y_delta_test, linear_pred_test)

print_metrics("Linear regression / train:", linear_train_metrics)
print_metrics("Linear regression / test:", linear_test_metrics)

**10) MLP на готовых данных:**

In [ ]:
# MLP работает с тем же плоским представлением окна, что и линейная модель
mlp_model = MLPRegressor(
    hidden_layer_sizes=(256, 128, 64),
    activation="relu",
    solver="adam",
    batch_size=128,
    learning_rate_init=1e-3,
    alpha=1e-4,
    max_iter=1,
    warm_start=True,
    random_state=RANDOM_STATE,
)

# история ошибки нужна для последующего графика loss
mlp_train_loss = []
mlp_test_loss = []

for epoch in range(MLP_EPOCHS):
    # обучаем одну эпоху и сразу пересчитываем качество на train/test
    mlp_model.fit(X_train, y_delta_train)

    pred_train = mlp_model.predict(X_train)
    pred_test = mlp_model.predict(X_test)

    mlp_train_loss.append(mean_squared_error(y_delta_train, pred_train))
    mlp_test_loss.append(mean_squared_error(y_delta_test, pred_test))

# финальные предсказания после завершения обучения
mlp_pred_train = mlp_model.predict(X_train)
mlp_pred_test = mlp_model.predict(X_test)

mlp_train_metrics = regression_metrics(y_delta_train, mlp_pred_train)
mlp_test_metrics = regression_metrics(y_delta_test, mlp_pred_test)

print_metrics("MLP / train:", mlp_train_metrics)
print_metrics("MLP / test:", mlp_test_metrics)

**11) LSTM на тензорах и последовательностях:**

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 128, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        # рекуррентный блок читает последовательность длины L
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )
        # голова преобразует скрытое состояние последнего шага в регрессионный прогноз
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        # используем состояние последнего временного шага как сводку окна
        last_hidden = out[:, -1, :]
        return self.head(last_hidden)

# инициализация модели, функции потерь и оптимизатора
lstm_model = LSTMRegressor(input_size=N_FEATURES_PER_STEP).to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE)

# отдельно собираем историю ошибки по эпохам
lstm_train_loss = []
lstm_test_loss = []

for epoch in range(LSTM_EPOCHS):
    lstm_model.train()

    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()
        pred = lstm_model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

    # после каждой эпохи оцениваем модель на всех train/test тензорах
    lstm_model.eval()
    with torch.no_grad():
        train_pred_tensor = lstm_model(X_train_tensor.to(DEVICE)).cpu()
        test_pred_tensor = lstm_model(X_test_tensor.to(DEVICE)).cpu()

        train_mse = criterion(train_pred_tensor, y_train_tensor).item()
        test_mse = criterion(test_pred_tensor, y_test_tensor).item()

    lstm_train_loss.append(train_mse)
    lstm_test_loss.append(test_mse)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(
            f"Epoch {epoch + 1:02d}/{LSTM_EPOCHS} | "
            f"train_mse={train_mse:.6f} | test_mse={test_mse:.6f}"
        )

# финальные numpy-массивы нужны для общей таблицы метрик
lstm_pred_train = train_pred_tensor.numpy().ravel()
lstm_pred_test = test_pred_tensor.numpy().ravel()

lstm_train_metrics = regression_metrics(y_delta_train, lstm_pred_train)
lstm_test_metrics = regression_metrics(y_delta_test, lstm_pred_test)

print_metrics("LSTM / train:", lstm_train_metrics)
print_metrics("LSTM / test:", lstm_test_metrics)

**12) Сводная таблица качества моделей:**

In [ ]:
# собираем метрики всех моделей в одну таблицу и сортируем по лучшему test_MAE
results_df = pd.DataFrame(
    [
        {
            "model": "LinearRegression_SGD",
            "train_MAE": linear_train_metrics["MAE"],
            "test_MAE": linear_test_metrics["MAE"],
            "train_MSE": linear_train_metrics["MSE"],
            "test_MSE": linear_test_metrics["MSE"],
            "train_RMSE": linear_train_metrics["RMSE"],
            "test_RMSE": linear_test_metrics["RMSE"],
            "train_R2": linear_train_metrics["R2"],
            "test_R2": linear_test_metrics["R2"],
        },
        {
            "model": "MLP",
            "train_MAE": mlp_train_metrics["MAE"],
            "test_MAE": mlp_test_metrics["MAE"],
            "train_MSE": mlp_train_metrics["MSE"],
            "test_MSE": mlp_test_metrics["MSE"],
            "train_RMSE": mlp_train_metrics["RMSE"],
            "test_RMSE": mlp_test_metrics["RMSE"],
            "train_R2": mlp_train_metrics["R2"],
            "test_R2": mlp_test_metrics["R2"],
        },
        {
            "model": "LSTM",
            "train_MAE": lstm_train_metrics["MAE"],
            "test_MAE": lstm_test_metrics["MAE"],
            "train_MSE": lstm_train_metrics["MSE"],
            "test_MSE": lstm_test_metrics["MSE"],
            "train_RMSE": lstm_train_metrics["RMSE"],
            "test_RMSE": lstm_test_metrics["RMSE"],
            "train_R2": lstm_train_metrics["R2"],
            "test_R2": lstm_test_metrics["R2"],
        },
    ]
).sort_values("test_MAE").reset_index(drop=True)

display(results_df)

# первая строка после сортировки считается лучшей моделью
best_model_name = results_df.iloc[0]["model"]
print("Лучшая модель по test_MAE:", best_model_name)

**13) Визуализация функции потерь (loss function):**

In [ ]:
# на одном графике сравниваем динамику ошибок у линейной модели и MLP
plt.figure(figsize=(12, 5))
plt.plot(linear_train_loss, label="Linear train loss")
plt.plot(linear_test_loss, label="Linear test loss")
plt.plot(mlp_train_loss, label="MLP train loss")
plt.plot(mlp_test_loss, label="MLP test loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Loss curves for Linear Regression and MLP")
plt.legend()
plt.grid(True)
plt.show()

# отдельно показываем loss-кривые LSTM, потому что их динамика обычно отличается
plt.figure(figsize=(12, 5))
plt.plot(lstm_train_loss, label="LSTM train loss")
plt.plot(lstm_test_loss, label="LSTM test loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Loss curves for LSTM")
plt.legend()
plt.grid(True)
plt.show()

**14) Визуальное сравнение ошибок моделей на тесте:**

In [ ]:
# сравнение по MAE: чем ниже столбец, тем лучше модель на тесте
plt.figure(figsize=(10, 5))
plt.bar(results_df["model"], results_df["test_MAE"])
plt.ylabel("Test MAE")
plt.title("Сравнение моделей по test MAE")
plt.grid(axis="y")
plt.show()

# сравнение по RMSE: эта метрика сильнее штрафует крупные ошибки
plt.figure(figsize=(10, 5))
plt.bar(results_df["model"], results_df["test_RMSE"])
plt.ylabel("Test RMSE")
plt.title("Сравнение моделей по test RMSE")
plt.grid(axis="y")
plt.show()

**15) Пример выбора лучшей модели и прогноз для преподавателя по 14 последним дням:**

In [ ]:
def prepare_student_window(student_14_days_df: pd.DataFrame) -> np.ndarray:
    '''
    На вход подается dataframe ровно по одному студенту за последние 14 дней.
    Возвращает:
      - flat-вектор формы (1, L*F) для Linear/MLP
      - последовательность формы (1, L, F) для LSTM можно получить reshape ниже.
    '''
    # без id пользователя и даты нельзя корректно собрать и упорядочить окно
    required_cols = {USER_COL, DATE_COL}
    missing_cols = required_cols - set(student_14_days_df.columns)
    if missing_cols:
        raise ValueError(f"Во входном dataframe не хватает колонок: {sorted(missing_cols)}")

    student_df = student_14_days_df.copy()
    student_df[DATE_COL] = pd.to_datetime(student_df[DATE_COL], errors="coerce")

    # функция рассчитана на одного студента; смешивать несколько пользователей в одном окне нельзя
    if student_df[USER_COL].nunique(dropna=True) > 1:
        raise ValueError("Для прогноза передайте данные только одного студента.")

    if student_df[DATE_COL].isna().any():
        raise ValueError("В колонке с датой есть некорректные значения, их нужно исправить перед прогнозом.")

    # сортируем записи по времени и оставляем последние L наблюдений
    student_df = student_df.sort_values(DATE_COL).reset_index(drop=True)

    if len(student_df) < L:
        raise ValueError(f"Для прогноза нужно минимум {L} записей (дней наблюдения).")

    # берем последние 14 дней
    student_df = student_df.tail(L).copy()

    # выделяем только те же признаки, что использовались при обучении
    cols_for_features = [c for c in num_cols if c in student_df.columns] + [c for c in cat_cols if c in student_df.columns]
    tmp = student_df[[USER_COL, DATE_COL] + cols_for_features].copy()

    if cat_cols:
        # one-hot должен повторять схему обучения; недостающие колонки добьем ниже нулями
        tmp = pd.get_dummies(tmp, columns=[c for c in cat_cols if c in tmp.columns], dummy_na=True)

    # добавляем недостающие признаки, которых могло не оказаться у конкретного студента
    for col in row_feature_cols:
        if col not in tmp.columns:
            tmp[col] = 0

    tmp = tmp[[USER_COL, DATE_COL] + row_feature_cols]

    # возвращаем плоское окно в том же масштабе, что использовался при обучении
    flat = tmp[row_feature_cols].to_numpy(dtype=np.float32).reshape(1, -1)
    flat = flat_scaler.transform(flat).astype(np.float32)
    return flat

def predict_next_14_days_growth(student_14_days_df: pd.DataFrame, current_fitness_level: float, model_name: str = None):
    # если модель явно не указана, берем лидера по test_MAE
    model_name = model_name or best_model_name

    if pd.isna(current_fitness_level):
        raise ValueError("current_fitness_level должен быть числом, а не NaN.")

    x_flat = prepare_student_window(student_14_days_df)

    if model_name == "LinearRegression_SGD":
        pred_delta = float(linear_model.predict(x_flat)[0])
    elif model_name == "MLP":
        pred_delta = float(mlp_model.predict(x_flat)[0])
    elif model_name == "LSTM":
        # для LSTM возвращаемся от плоского представления к трехмерному тензору
        x_seq = x_flat.reshape(1, L, N_FEATURES_PER_STEP)
        x_tensor = torch.tensor(x_seq, dtype=torch.float32).to(DEVICE)
        lstm_model.eval()
        with torch.no_grad():
            pred_delta = float(lstm_model(x_tensor).cpu().numpy().ravel()[0])
    else:
        raise ValueError(f"Неизвестная модель: {model_name}")

    # абсолютный прогноз на 14 дней = текущий уровень + предсказанное изменение
    pred_future_fitness = current_fitness_level + pred_delta

    return {
        "model_used": model_name,
        "current_fitness_level": float(current_fitness_level),
        "predicted_delta_14_days": pred_delta,
        "predicted_fitness_level_after_14_days": float(pred_future_fitness),
    }

print(
    "Функция predict_next_14_days_growth(...) готова.\n"
    "Передайте в неё dataframe с 14 последними днями конкретного студента и текущее fitness_level."
)